# 11 - Fine-tuning TripoSR on sketches

Runs on Ibex through the VS Code Jupyter server, on one A100.

Six sections, one job each: **setup**, **data**, **dataset**, **dataloaders**, **model**, **train**, **test**.

The recipe follows TripoSR's report where the report actually says something: MSE + 2.0·LPIPS + 0.05·mask-BCE on 128px foreground-biased crops, AdamW into a cosine schedule after a warmup, no camera conditioning, and rendering losses alone as supervision. It differs deliberately in two places — the whole network is fine-tuned rather than adapted, and it is unfrozen **gradually**, outermost layers first, so the pretrained encoder is not destroyed by the large early gradients of a task it has never seen.

**On the 13.** The paper conditions on **one** image, and so does this notebook — which is also what inference does, so training and deployment see the same thing. Its loss (Eq. 2) averages over `V` *supervision* views, and the report never states V's value. There is no "13 conditioning images" in it: every occurrence of `13` in the paper is the citation `[13]`, which is ZeroShape. So 13 lives in the one place it legitimately can — **13 supervision views per sample**, out of the 16 cameras each design was rendered from, the input's own view among them.

Notebook 08 pulls the data from Drive to Ibex; notebook 10 builds the sketch shards. If the sketch shards never made it here, the dataset falls back to edge-detecting each render as it loads and the run proceeds unchanged.

In [ ]:
# Setup. Ibex, one A100: locate the repo, turn on the fast matmul paths,
# and report the node. `git -C $HOME/StarX pull` first to pick up changes.
import os
import subprocess
import sys


def _find_repo():
    for start in (globals().get("__vsc_ipynb_file__"), os.getcwd()):
        if not start:
            continue
        path = os.path.abspath(
            os.path.dirname(start) if os.path.isfile(start) else start
        )
        while path != os.path.dirname(path):
            if os.path.exists(os.path.join(path, "starx", "pins.py")):
                return path
            path = os.path.dirname(path)
    home = os.path.join(os.path.expanduser("~"), "StarX")
    if os.path.exists(os.path.join(home, "starx", "pins.py")):
        return home
    raise RuntimeError("could not locate the StarX repo - clone it to ~/StarX")


REPO_DIR = _find_repo()
TRIPOSR_DIR = os.path.join(REPO_DIR, "third_party", "TripoSR")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch

# TF32 costs nothing on an A100 and speeds up every matmul in the encoder
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print("repo:  ", REPO_DIR)
print("commit:", subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                                capture_output=True, text=True).stdout.strip())
print("torch: ", torch.__version__)
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    cap = torch.cuda.get_device_capability(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  sm_{cap[0]}{cap[1]}  "
          f"{free / 2**30:.0f}/{total / 2**30:.0f} GiB free")
assert torch.cuda.is_available(), "no GPU visible - are you on a compute node?"
assert os.path.exists(TRIPOSR_DIR), f"TripoSR clone missing at {TRIPOSR_DIR}"

In [ ]:
# Configuration - every tunable for this notebook lives here.
import json
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchmetrics.image import LearnedPerceptualImagePatchSimilarity
from tqdm.auto import tqdm

from starx import cameras, checkpoint, data, shards, sketchdata, synth
from starx import model as smodel
from starx import train as strain
from starx.config import (
    CAMERA_DISTANCE, FOVY_DEG, StarXConfig, run_dir, shard_dir, sketch_shard_dir,
)

CFG = {
    "run_name": "sketch_ft",
    "resume": True,
    "seed": 1337,
    # data
    "supervision_views": 13,   # views scored per sample; the input is one of them
    "batch": 4,                # designs per step -> batch x views crop renders
    "workers": 8,
    # schedule
    "epochs": 8,
    "total_steps": None,       # None: epochs x batches
    "warmup": 500,
    # optimization
    "lr": 1e-4,                # decoder rate; earlier groups get a fraction
    "weight_decay": 0.05,
    "betas": (0.9, 0.95),
    "grad_clip": 1.0,
    # loss (TripoSR's stated weights)
    "lambda_lpips": 2.0,
    "lambda_mask": 0.05,
    "crop": 128,
    # memory / speed
    "grad_checkpoint": False,  # False is faster; flip it if the step OOMs
    "ckpt_every": 500,
    "val_every": 500,
    "keep_k": 2,
}

cfg = StarXConfig(
    drive_root=str(Path(REPO_DIR) / "data" / "StarX"),
    local_root=str(Path(REPO_DIR) / "data" / "local"),
    sketch_size=512, edge_blur_sigma=1.2, edge_gain=3.0, edge_bg=1.0,
    lambda_mse=1.0, lambda_lpips=CFG["lambda_lpips"], lambda_mask=CFG["lambda_mask"],
    lambda_occ=0.0, render_crop=CFG["crop"], composite_bg=0.5,
    gt_size=256, eval_chunk=131072, seed=CFG["seed"],
)

DEVICE = "cuda:0"
AMP_DTYPE, _ = strain.pick_amp(DEVICE)
BG = cfg.composite_bg
EVAL_CHUNK = cfg.eval_chunk
RUN_DIR = run_dir(cfg, CFG["run_name"])
LOG_PATH = RUN_DIR / "logs" / "train_log.jsonl"
GRID_DIR = RUN_DIR / "val_grids"
GRID_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "logs").mkdir(parents=True, exist_ok=True)
crop_rng = np.random.default_rng(CFG["seed"])
colors = plt.get_cmap("tab10").colors

torch.manual_seed(CFG["seed"])
print(f"run {CFG['run_name']}  device {DEVICE}  autocast {AMP_DTYPE}")
print(f"{CFG['batch']} designs x {CFG['supervision_views']} views per step, "
      f"{CFG['crop']}px crops")
print(f"loss: MSE + {cfg.lambda_lpips} LPIPS + {cfg.lambda_mask} mask-BCE")
print(f"run dir: {RUN_DIR}")

## Data

In [ ]:
# Unpack the shards into a local cache. This is the ONLY place data enters
# the notebook, and it comes from what notebook 08 pulled off Drive into
# /ibex/user/$USER/StarX (symlinked to <repo>/data/StarX). Two shard sets
# land in the same cache: the design shards from notebook 03 and, if they
# were built and transferred, the sketch shards from notebook 10.
TRAIN_CACHE = Path(cfg.local_root) / "train" / "cache"
TEST_CACHE = Path(cfg.local_root) / "test" / "cache"

for split in ("train", "test"):
    local = Path(cfg.local_root) / split
    designs = shard_dir(cfg, split)
    assert shards.list_done_shards(designs), (
        f"no design shards at {designs}\n"
        f"  -> run notebook 08 to pull them from Drive, and check the symlink:\n"
        f"     ln -s /ibex/user/$USER/StarX  {REPO_DIR}/data/StarX"
    )
    shards.prepare_local(designs, local, progress=tqdm)

    sketches = sketch_shard_dir(cfg, split)
    if shards.list_done_shards(sketches, sketchdata.SKETCH_PREFIX):
        sketchdata.check_params(sketches, cfg)
        shards.prepare_local(sketches, local, progress=tqdm,
                             prefix=sketchdata.SKETCH_PREFIX)
        source = "prebuilt shards"
    else:
        source = "computed on the fly (no sketch_shards here)"
    n_designs = len(list((local / "cache").glob("*.meta.json")))
    print(f"{split:6s} {n_designs:5d} designs   sketches: {source}")

print(f"\ncache: {TRAIN_CACHE.parent.parent}")
print("if sketches say 'on the fly', notebook 08's INCLUDE_SKETCHES pull found "
      "nothing on Drive - training still works, it just re-derives each drawing")

## Dataset

In [ ]:
# The dataset. One sample = one (design, input view) pair, so each design
# contributes as many samples as it has cameras. A sample carries ONE
# sketch - what the model is conditioned on, and what inference will give
# it - plus V ground-truth views to be scored against, the first of which
# is always the input's own view (LRM's "input view plus side views").
#
# Cameras come back rotated so the input view sits at azimuth zero, because
# TripoSR reconstructs in the frame of the picture it was given. Sketches
# are read from the prebuilt shards when they are there and edge-detected
# on the fly when they are not - the two agree to within a uint8 level.
class SketchViewDataset(Dataset):
    def __init__(self, cache_dir, design_ids=None, supervision_views=13,
                 seed=1337, sketches="auto"):
        self.cache = Path(cache_dir)
        self.V = supervision_views
        self.seed = seed
        self.epoch = 0
        ids = design_ids if design_ids is not None else [
            p.name[: -len(".meta.json")] for p in sorted(self.cache.glob("*.meta.json"))
        ]
        self.designs = [str(d) for d in ids]
        self.meta = {d: json.loads((self.cache / f"{d}.meta.json").read_text())
                     for d in self.designs}
        self.n_views = min(m["n_views"] for m in self.meta.values())
        if sketches == "auto":
            probe = self.cache / sketchdata.sketch_member_name(self.designs[0], 0)
            sketches = "stored" if probe.exists() else "live"
        self.sketches = sketches

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.designs) * self.n_views

    def _sketch(self, design_id, view):
        if self.sketches == "stored":
            return sketchdata.load_sketch(self.cache, design_id, view)
        rgba = np.asarray(Image.open(self.cache / f"{design_id}.view{view:02d}.png"))
        return synth.sobel_sketch(rgba[..., :3], cfg.sketch_size, cfg.edge_blur_sigma,
                                  cfg.edge_gain, cfg.edge_bg)

    def __getitem__(self, index):
        d_index, input_view = divmod(index, self.n_views)
        design_id = self.designs[d_index]
        meta = self.meta[design_id]

        # supervision views: the input's own, then V-1 others, redrawn each epoch
        rng = np.random.default_rng([self.seed, index, self.epoch])
        others = [v for v in range(self.n_views) if v != input_view]
        picked = [input_view] + list(
            rng.choice(others, size=min(self.V - 1, len(others)), replace=False)
        )

        rgbs, masks = [], []
        for v in picked:
            rgba = np.asarray(Image.open(self.cache / f"{design_id}.view{v:02d}.png"))
            rgbs.append(rgba[..., :3])
            masks.append(rgba[..., 3] > 127)

        c2ws = np.load(self.cache / f"{design_id}.cameras.npy")
        c2ws = synth.canonicalize_to_view(
            c2ws, float(meta["view_angles"][input_view][0])
        )
        return {
            "input": self._sketch(design_id, input_view),      # (3, S, S)
            "views": torch.from_numpy(np.stack(rgbs)),          # (V, H, W, 3) uint8
            "masks": torch.from_numpy(np.stack(masks)),         # (V, H, W) bool
            "c2ws": torch.from_numpy(c2ws[picked].copy()),      # (V, 4, 4)
            "design_id": design_id,
            "input_view": int(input_view),
        }


def collate(items):
    out = {}
    for key in items[0]:
        values = [it[key] for it in items]
        out[key] = torch.stack(values) if torch.is_tensor(values[0]) else values
    return out


# design-level split: validation designs are never trained on
all_ids = sorted(p.name[: -len(".meta.json")] for p in TRAIN_CACHE.glob("*.meta.json"))
split_rng = np.random.default_rng(CFG["seed"])
shuffled = [str(d) for d in split_rng.permutation(all_ids)]
n_val = max(1, int(0.05 * len(shuffled)))
val_ids, train_ids = shuffled[:n_val], shuffled[n_val:]

kwargs = dict(supervision_views=CFG["supervision_views"], seed=CFG["seed"])
train_ds = SketchViewDataset(TRAIN_CACHE, train_ids, **kwargs)
val_ds = SketchViewDataset(TRAIN_CACHE, val_ids, **kwargs)
test_ds = SketchViewDataset(TEST_CACHE, **kwargs)

for name, ds in (("train", train_ds), ("val", val_ds), ("test", test_ds)):
    print(f"{name:6s} {len(ds.designs):5d} designs x {ds.n_views} views "
          f"= {len(ds):6d} samples   sketches: {ds.sketches}")

In [ ]:
# One sample, drawn: the single sketch the model is conditioned on, and
# the V ground-truth views its prediction is scored against.
sample = train_ds[0]
n = sample["views"].shape[0]
fig = plt.figure(figsize=(14, 4.4))
grid = fig.add_gridspec(2, max(7, (n + 1) // 2 + 1), hspace=0.25, wspace=0.08)

ax = fig.add_subplot(grid[:, 0])
ax.imshow(sample["input"][0], cmap="gray", vmin=0, vmax=1)
ax.set_title(f"INPUT sketch\nview {sample['input_view']}", fontsize=9)
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_edgecolor(colors[3]); spine.set_linewidth(2.5)

for i in range(n):
    r, c = divmod(i, max(7, (n + 1) // 2 + 1) - 1)
    ax = fig.add_subplot(grid[r, 1 + c])
    view = sample["views"][i].numpy().astype(np.float32) / 255.0
    ax.imshow(np.where(sample["masks"][i].numpy()[..., None], view, BG))
    ax.set_title(f"sup {i}", fontsize=7)
    ax.axis("off")
fig.suptitle(f"{sample['design_id']}  -  1 sketch in, {n} views supervised",
             fontsize=11)
plt.show()

## Dataloaders

In [ ]:
# Loaders. Workers do the PNG decoding so the GPU never waits on disk;
# the throughput number below should comfortably exceed the step rate
# measured later, otherwise raise CFG["workers"].
train_loader = DataLoader(
    train_ds, batch_size=CFG["batch"], shuffle=True,
    num_workers=CFG["workers"], collate_fn=collate, drop_last=True,
    pin_memory=True, persistent_workers=CFG["workers"] > 0,
    prefetch_factor=4 if CFG["workers"] > 0 else None,
)
val_loader = DataLoader(
    val_ds, batch_size=CFG["batch"], shuffle=False,
    num_workers=2, collate_fn=collate, pin_memory=True,
)
TOTAL_STEPS = CFG["total_steps"] or len(train_loader) * CFG["epochs"]

t0 = time.time()
batch = next(iter(train_loader))
first = time.time() - t0
t0 = time.time()
for i, _ in zip(range(5), train_loader):
    pass
rate = 5 * CFG["batch"] / (time.time() - t0)

print(f"train {len(train_loader)} batches/epoch, val {len(val_loader)}, "
      f"{TOTAL_STEPS} steps total ({CFG['epochs']} epochs)")
print(f"first batch {first:.1f}s, then ~{rate:.0f} samples/s from {CFG['workers']} workers")
for k, v in batch.items():
    print(f"  {k:11s} {tuple(v.shape) if torch.is_tensor(v) else type(v).__name__}"
          f" {v.dtype if torch.is_tensor(v) else ''}")

fig, axes = plt.subplots(1, CFG["batch"], figsize=(2.0 * CFG["batch"], 2.4))
for ax, img, did in zip(np.atleast_1d(axes), batch["input"], batch["design_id"]):
    ax.imshow(img[0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(did[:10], fontsize=7)
    ax.axis("off")
fig.suptitle("one batch of inputs", fontsize=10)
fig.tight_layout()
plt.show()

## Model

In [ ]:
# Stock TripoSR, straight off the shelf: three input channels, pretrained
# normalization, nothing inflated or replaced. Only the weights will move.
model = smodel.load_pretrained_tsr(TRIPOSR_DIR, device=DEVICE)
projection = model.image_tokenizer.model.embeddings.patch_embeddings.projection
assert projection.in_channels == 3, "this notebook fine-tunes the unmodified model"
if CFG["grad_checkpoint"]:
    model.image_tokenizer.model.gradient_checkpointing_enable()
    model.backbone.gradient_checkpointing = True
model.renderer.set_chunk_size(0)

table = smodel.param_count_table(model)
parts = {k: v["total"] / 1e6 for k, v in table.items() if k != "ALL"}
print(f"{'module':18s} {'params (M)':>11s}")
for name, millions in sorted(parts.items(), key=lambda kv: -kv[1]):
    print(f"{name:18s} {millions:>11.1f}")
print(f"{'TOTAL':18s} {table['ALL']['total'] / 1e6:>11.1f}")

fig, ax = plt.subplots(figsize=(9, 2.6))
names = sorted(parts, key=lambda k: -parts[k])
ax.bar(names, [parts[n] for n in names], color=colors[: len(names)])
ax.set_ylabel("params (M)")
ax.set_title("where TripoSR's parameters live", fontsize=10)
ax.tick_params(axis="x", rotation=20, labelsize=8)
for i, n in enumerate(names):
    ax.text(i, parts[n], f"{parts[n]:.0f}", ha="center", va="bottom", fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# Gradual unfreezing. Everything starts frozen except the pieces closest to
# the output; each stage unlocks one layer deeper into the network. The
# reason is that at step zero the decoder is being asked to explain a kind
# of image it has never seen, so its gradients are large and noisy - and
# letting those reach DINO immediately is how a pretrained encoder gets
# wrecked in the first hundred steps.
STAGES = [
    {"at": 0.00, "name": "head",       "groups": ["decoder", "post_processor"]},
    {"at": 0.10, "name": "+triplane",  "groups": ["tokenizer"]},
    {"at": 0.25, "name": "+backbone",  "groups": ["backbone"]},
    {"at": 0.55, "name": "+encoder",   "groups": ["image_tokenizer"]},
]
stage_boundaries = [int(s["at"] * TOTAL_STEPS) for s in STAGES]
stage_colors = [colors[i] for i in range(len(STAGES))]
current_stage_name = None


def trainable():
    return [p for p in model.parameters() if p.requires_grad]


def apply_stage(step):
    """Freeze everything, then unlock every group whose stage has arrived."""
    global current_stage_name
    active = [s for s, b in zip(STAGES, stage_boundaries) if step >= b]
    name = active[-1]["name"] if active else STAGES[0]["name"]
    if name == current_stage_name:
        return False
    for p in model.parameters():
        p.requires_grad_(False)
    for stage in active:
        for group in stage["groups"]:
            getattr(model, group).requires_grad_(True)
    current_stage_name = name
    return True


# what the schedule looks like, as trainable parameters over the run
curve, marks = [], []
for step in range(0, TOTAL_STEPS, max(1, TOTAL_STEPS // 400)):
    apply_stage(step)
    curve.append((step, sum(p.numel() for p in trainable()) / 1e6))
apply_stage(0)

fig, ax = plt.subplots(figsize=(11, 3.0))
xs, ys = zip(*curve)
ax.step(xs, ys, where="post", color=colors[0], lw=2)
ax.fill_between(xs, ys, step="post", alpha=0.15, color=colors[0])
for i, (b, stage) in enumerate(zip(stage_boundaries, STAGES)):
    ax.axvline(b, color=stage_colors[i], ls="--", lw=1.2)
    ax.annotate(f"{stage['name']}\n{'+'.join(stage['groups'])}",
                (b, max(ys) * (0.82 - 0.22 * (i % 2))), fontsize=8,
                color=stage_colors[i])
ax.set_xlabel("step")
ax.set_ylabel("trainable (M)")
ax.set_title("gradual unfreezing", fontsize=10)
ax.grid(alpha=0.3)
plt.show()
print(f"stages open at steps: {stage_boundaries}  of {TOTAL_STEPS}")

In [ ]:
# AdamW with DISCRIMINATIVE learning rates: the deeper into pretraining a
# group sits, the less it moves. The decoder is closest to our new task and
# gets the full rate; DINO saw millions of photographs and gets a tenth of
# it. Weight decay skips biases and normalization scales.
LR_SCALE = {"decoder": 1.0, "post_processor": 1.0, "tokenizer": 0.5,
            "backbone": 0.5, "image_tokenizer": 0.1}

groups = []
for name, scale in LR_SCALE.items():
    module = getattr(model, name)
    decay = [p for p in module.parameters() if p.ndim > 1]
    plain = [p for p in module.parameters() if p.ndim <= 1]
    groups.append({"params": decay, "lr": CFG["lr"] * scale,
                   "weight_decay": CFG["weight_decay"], "name": f"{name}"})
    groups.append({"params": plain, "lr": CFG["lr"] * scale,
                   "weight_decay": 0.0, "name": f"{name}.nodecay"})

optimizer = torch.optim.AdamW(groups, lr=CFG["lr"], betas=CFG["betas"])


def lr_lambda(s):
    if s < CFG["warmup"]:
        return (s + 1) / CFG["warmup"]
    progress = (s - CFG["warmup"]) / max(1, TOTAL_STEPS - CFG["warmup"])
    return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

lpips_metric = LearnedPerceptualImagePatchSimilarity(net_type="vgg", normalize=True)
lpips_metric = lpips_metric.to(DEVICE).requires_grad_(False).eval()

print(f"{'group':18s} {'lr':>10s}  params")
for g in optimizer.param_groups:
    if g["params"]:
        print(f"{g['name']:18s} {g['lr']:>10.2e}  "
              f"{sum(p.numel() for p in g['params']):,}")

fig, ax = plt.subplots(figsize=(11, 2.6))
schedule = [lr_lambda(s) * CFG["lr"] for s in range(TOTAL_STEPS)]
ax.plot(schedule, color=colors[4], lw=1.5)
for i, b in enumerate(stage_boundaries):
    ax.axvline(b, color=stage_colors[i], ls="--", lw=1.2)
    ax.annotate(STAGES[i]["name"], (b, max(schedule) * (0.9 - 0.12 * (i % 3))),
                fontsize=8, color=stage_colors[i])
ax.set_xlabel("step")
ax.set_ylabel("lr (decoder)")
ax.set_title("learning rate and unfreeze points", fontsize=10)
ax.grid(alpha=0.3)
plt.show()

## Train

In [ ]:
# One optimizer step, written out. Encode the batch's sketches once, then
# for each design render V crops at the ground-truth cameras and score
# them. Every view backpropagates only as far as a detached copy of the
# scene code, which frees its render graph immediately; the accumulated
# gradient then goes through the encoder once for the whole batch. That is
# the deferred backpropagation the LRM line of work relies on, and it is
# what makes 13 supervision views fit at all.
def score_view(code, batch, b, v, full_frame=False):
    """Render one view of one design and return its loss terms."""
    views, masks = batch["views"][b].numpy(), batch["masks"][b].numpy()
    c2w, size = batch["c2ws"][b][v].numpy(), views.shape[1]
    if full_frame:
        rays_o, rays_d = cameras.rays_full(c2w, FOVY_DEG, size)
        top, left, h, w = 0, 0, size, size
    else:
        top, left, h, w = data.sample_crop_box(masks[v], CFG["crop"], crop_rng)
        rays_o, rays_d = cameras.rays_for_crop(
            c2w, FOVY_DEG, size, (top, left, h, w)
        )
    rgb, alpha = strain.render_rays(model, code, rays_o.to(DEVICE), rays_d.to(DEVICE))
    gt_rgb = torch.from_numpy(
        np.ascontiguousarray(views[v][top:top + h, left:left + w])
    ).float().to(DEVICE) / 255.0
    gt_mask = torch.from_numpy(
        np.ascontiguousarray(masks[v][top:top + h, left:left + w])
    ).to(DEVICE)
    loss, parts = strain.compute_render_loss(
        rgb, alpha, gt_rgb, gt_mask, lpips_metric, cfg
    )
    return (loss, parts) if not full_frame else parts


def train_step(batch, step):
    model.train()
    inputs = batch["input"].to(DEVICE, non_blocking=True)
    B, V = inputs.shape[0], batch["views"].shape[1]
    n_terms = B * V
    totals = {"loss": 0.0, "mse": 0.0, "lpips": 0.0, "mask": 0.0}

    with torch.autocast("cuda", dtype=AMP_DTYPE):
        codes = smodel.encode_sketches(model, inputs)
    codes = codes.float()

    code_grads = torch.zeros_like(codes)
    for b in range(B):
        leaf = codes[b].detach().requires_grad_(True)
        for v in range(V):
            loss, parts = score_view(leaf, batch, b, v)
            (loss / n_terms).backward()      # frees this view's render graph
            totals["loss"] += float(loss.detach()) / n_terms
            for k in ("mse", "lpips", "mask"):
                totals[k] += parts[k] / n_terms
        code_grads[b] = leaf.grad
    codes.backward(gradient=code_grads)       # one encoder backward per step

    grad_norm = torch.nn.utils.clip_grad_norm_(trainable(), CFG["grad_clip"])
    if torch.isfinite(grad_norm):
        optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    totals["grad_norm"] = float(grad_norm)
    totals["lr"] = scheduler.get_last_lr()[0]
    return totals


# smoke it once: this is where a speed or memory problem shows up
torch.cuda.reset_peak_memory_stats()
probe_batch = next(iter(train_loader))
apply_stage(0)
t0 = time.time()
probe = train_step(probe_batch, 0)
elapsed = time.time() - t0
peak = torch.cuda.max_memory_allocated() / 2**30

print(f"one step: {elapsed:.2f}s   peak VRAM {peak:.1f} GiB   loss {probe['loss']:.3f}")
print(f"  {CFG['batch']} designs x {CFG['supervision_views']} views "
      f"= {CFG['batch'] * CFG['supervision_views']} crop renders")
print(f"projected: {elapsed * TOTAL_STEPS / 3600:.1f} h for {TOTAL_STEPS} steps")
if peak > 60:
    print("  tight on memory - set CFG['grad_checkpoint']=True or lower the batch")

In [ ]:
# Resume, and the visual the run reports with: input | ground truth |
# prediction from the input camera | prediction turned a quarter turn.
# The turned column is the one that matters - a flat billboard facing the
# input camera looks perfect in column 3 and falls apart in column 4.
latest = checkpoint.find_latest(RUN_DIR) if CFG["resume"] else None
if latest is not None:
    ckpt_path, start_step = latest
    state = checkpoint.load_checkpoint(ckpt_path)
    apply_stage(start_step)   # rebuild the same trainable set before loading
    smodel.load_trainable_state_dict(model, state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    checkpoint.restore_rng(state["rng"])
    print(f"resumed {CFG['run_name']} at step {start_step}")
else:
    start_step = 0
    print(f"starting {CFG['run_name']} fresh")

NOVEL_C2W = cameras.build_spherical_c2w(90.0, 20.0, CAMERA_DISTANCE)


def gallery(dataset, title, n=3):
    """One row per design: sketch, ground truth, prediction, prediction turned."""
    model.eval()
    model.renderer.set_chunk_size(EVAL_CHUNK)
    rows = [dataset[i * dataset.n_views] for i in range(min(n, len(dataset.designs)))]
    fig, axes = plt.subplots(len(rows), 4, figsize=(12.4, 3.1 * len(rows)))
    axes = np.atleast_2d(axes)
    with torch.no_grad():
        for r, item in enumerate(rows):
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                code = smodel.encode_sketches(model, item["input"][None].to(DEVICE))[0]
            code = code.float()
            size = item["views"].shape[1]
            preds = []
            for c2w in (item["c2ws"][0].numpy(), NOVEL_C2W):
                rays_o, rays_d = cameras.rays_full(c2w, FOVY_DEG, size)
                rgb, alpha = strain.render_rays(
                    model, code, rays_o.to(DEVICE), rays_d.to(DEVICE)
                )
                preds.append(
                    strain.composite_over_gray(rgb, alpha, BG).clamp(0, 1).cpu().numpy()
                )
            gt = item["views"][0].numpy().astype(np.float32) / 255.0
            gt = np.where(item["masks"][0].numpy()[..., None], gt, BG)
            for c, img in enumerate([item["input"][0].numpy(), gt, preds[0], preds[1]]):
                axes[r, c].imshow(img, cmap="gray" if img.ndim == 2 else None,
                                  vmin=0, vmax=1)
            axes[r, 0].set_ylabel(item["design_id"][:11], fontsize=7)
    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
    for c, t in enumerate(["sketch in", "ground truth", "prediction", "turned 90 deg"]):
        axes[0, c].set_title(t, fontsize=10)
    fig.suptitle(title, fontsize=11, y=1.0)
    fig.tight_layout()
    model.renderer.set_chunk_size(0)
    model.train()
    return fig


fig = gallery(val_ds, "before training", n=3)
plt.show()

In [ ]:
# The run. Resumable - re-running picks up from the newest checkpoint.
step = start_step
epoch_offset = step // max(1, len(train_loader))
progress = tqdm(total=TOTAL_STEPS, initial=step, desc="train")
timer = time.time()

while step < TOTAL_STEPS:
    epoch = epoch_offset + (step - start_step) // max(1, len(train_loader))
    train_ds.set_epoch(epoch)
    for batch in train_loader:
        if step >= TOTAL_STEPS:
            break
        apply_stage(step)              # gradual unfreezing happens here
        totals = train_step(batch, step)
        step += 1
        progress.update(1)
        progress.set_postfix(
            loss=f"{totals['loss']:.3f}", stage=current_stage_name,
            **{"s/it": f"{(time.time() - timer) / max(1, step - start_step):.2f}"},
        )
        if step % 25 == 0 or step == TOTAL_STEPS:
            checkpoint.append_log(LOG_PATH, {"step": step, "epoch": epoch, **totals})
        if step % CFG["ckpt_every"] == 0 or step == TOTAL_STEPS:
            checkpoint.save_checkpoint(
                RUN_DIR, step,
                {
                    "model": smodel.trainable_state_dict(model),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "rng": checkpoint.rng_states(),
                },
                keep_k=CFG["keep_k"],
            )
        if step % CFG["val_every"] == 0 or step == TOTAL_STEPS or step == start_step + 1:
            fig = gallery(val_ds, f"validation @ step {step}", n=3)
            fig.savefig(GRID_DIR / f"step_{step:07d}.png", dpi=100, bbox_inches="tight")
            plt.show()
progress.close()
print(f"done: {TOTAL_STEPS - start_step} steps in "
      f"{(time.time() - timer) / 3600:.2f} h")

In [ ]:
# How the run went: the loss terms, the learning rate, and the stage
# boundaries drawn on top so a jump at an unfreeze is obvious.
history = checkpoint.read_log(LOG_PATH)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
colors = plt.get_cmap("tab10").colors

for ax, keys, title in (
    (axes[0], ["loss"], "total loss"),
    (axes[1], ["mse", "lpips", "mask"], "loss terms"),
    (axes[2], ["lr"], "learning rate"),
):
    for i, key in enumerate(keys):
        if key in history:
            ax.plot(history["step"], history[key], color=colors[i], label=key, lw=1.2)
    for boundary in stage_boundaries[1:]:
        ax.axvline(boundary, color="0.7", ls="--", lw=1)
    ax.set_xlabel("step")
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    if title != "learning rate":
        ax.set_yscale("log")
axes[0].annotate("dashed = unfreeze", (0.02, 0.06), xycoords="axes fraction", fontsize=8)
fig.tight_layout()
plt.show()

grids = sorted(GRID_DIR.glob("step_*.png"))
if len(grids) >= 2:
    fig, axes = plt.subplots(2, 1, figsize=(13, 11))
    for ax, path in zip(axes, [grids[0], grids[-1]]):
        ax.imshow(Image.open(path))
        ax.set_title(path.stem, fontsize=9)
        ax.axis("off")
    fig.tight_layout()
    plt.show()

## Test

In [ ]:
# Held-out test split: the same picture on designs the model never trained
# on. This is the number to quote, not the training loss.
test_loader = DataLoader(
    test_ds, batch_size=CFG["batch"], shuffle=False,
    num_workers=CFG["workers"], collate_fn=collate, pin_memory=True,
)

model.eval()
model.renderer.set_chunk_size(EVAL_CHUNK)
totals, n_batches = {"mse": 0.0, "lpips": 0.0, "mask": 0.0}, 0
with torch.no_grad():
    for batch in tqdm(test_loader, desc="test"):
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            codes = smodel.encode_sketches(model, batch["input"].to(DEVICE))
        codes = codes.float()
        for b in range(codes.shape[0]):
            for v in range(min(2, batch["views"].shape[1])):  # 2 views is enough to rank
                parts = score_view(codes[b], batch, b, v, full_frame=True)
                for k in totals:
                    totals[k] += parts[k]
        n_batches += codes.shape[0] * min(2, batch["views"].shape[1])
model.renderer.set_chunk_size(0)

print(f"held-out test over {len(test_ds)} samples "
      f"({len(test_ds.designs)} designs never seen in training):")
for k, v in totals.items():
    print(f"  {k:6s} {v / n_batches:.4f}")

fig = gallery(test_ds, "held-out test designs", n=4)
plt.show()